In [0]:
%python
from enum import Enum
from typing import List
from pydantic import BaseModel, Field, ValidationError, field_validator
from pprint import pprint


In [0]:
%python
class PriorityEnum(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

In [0]:
class TicketAction(BaseModel):
    action_type: str = Field(description="The action to execute, e.g., 'escalate' or 'refund'")
    target_department: str = Field(description="Department responsible for execution")

In [0]:
class AgentAnalysisOutput(BaseModel):
    ticket_id: int = Field(gt=0, description="Unique positive integer ticket ID")
    summary: str = Field(min_length=10, description="A concise summary of the issue")
    priority: PriorityEnum = Field(description="Assigned issue priority level")
    confidence_score: float = Field(ge=0.0, le=1.0, description="Model confidence score between 0.0 and 1.0")
    proposed_actions: List[TicketAction] = Field(min_length=1, description="List of actions to take")
    
    @field_validator("summary")
    @classmethod
    def summary_must_be_descriptive(cls, value: str) -> str:
        if len(value.split()) < 3:
            raise ValueError("Summary is too short; provide at least 3 words.")
        return value


# --- Demonstration ---

In [0]:
# Simulated JSON dictionary returned by an LLM
raw_llm_json = {
    "ticket_id": "1042",  # String "1042" will be automatically coerced to int 1042
    "summary": "Customer unable to reset password via email link.",
    "priority": "high",
    "confidence_score": 0.95,
    "proposed_actions": [
        {"action_type": "send_reset_link", "target_department": "Authentication"}
    ]
}

In [0]:
# Simulated JSON dictionary returned by an LLM
raw_llm_json_1 = {
    "ticket_id": "1042",  # String "1042" will be automatically coerced to int 1042
    "summary": "Customer unable ",
    "priority": "high",
    "confidence_score": 0.95,
    "proposed_actions": [
        {"action_type": "send_reset_link", "target_department": "Authentication"}
    ]
}

In [0]:
parsed_output = AgentAnalysisOutput.model_validate(raw_llm_json_1)
print(parsed_output)

In [0]:
# Parse and validate raw data into Python objects
try:
    parsed_output = AgentAnalysisOutput.model_validate(raw_llm_json_1)
    print(f"Validated Ticket ID: {parsed_output.ticket_id} (Type: {type(parsed_output.ticket_id).__name__})")
    print(f"Action: {parsed_output.proposed_actions[0].action_type}")
    
    # Export back to JSON for downstream APIs or database storage
    print("\nExported JSON Schema Output:")
    print(parsed_output.model_dump_json(indent=2))

except ValidationError as e:
    print("Validation Error:", e)